In [1]:
# =============================================================================
# Insider Threat Behavioral Intelligence System
# Notebook : 03_model_comparison.ipynb
# =============================================================================

# Module 03: Comparative Analysis & Model Benchmarking

This notebook evaluates overlap, correlation, similarity, and anomaly detection performance across all 7 trained models:
- Anomaly Yield & Frequency Analysis
- Model Jaccard Overlap Matrix
- Model Anomaly Score Correlation Heatmap
- Consensus Anomaly Detection Matrix

In [2]:
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)

print("Visualization libraries loaded successfully!")

Visualization libraries loaded successfully!


In [3]:
PROJECT_ROOT = Path("..").resolve()
PREDICTION_FILE = PROJECT_ROOT / "datasets" / "predictions" / "all_model_predictions.parquet"
REPORT_DIR = PROJECT_ROOT / "reports"
PLOT_DIR = PROJECT_ROOT / "plots"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

if PREDICTION_FILE.exists():
    df_preds = pd.read_parquet(PREDICTION_FILE)
elif (PROJECT_ROOT / "datasets" / "predictions" / "all_model_predictions.csv").exists():
    df_preds = pd.read_csv(PROJECT_ROOT / "datasets" / "predictions" / "all_model_predictions.csv")
else:
    print("Generating simulated prediction dataset for model comparison demonstration.")
    np.random.seed(42)
    n = 1000
    df_preds = pd.DataFrame({'user': [f'USR{i:04d}' for i in range(n)]})
    models = ['isolation_forest', 'one_class_svm', 'lof', 'elliptic_envelope', 'pca', 'dbscan', 'kmeans']
    for m in models:
        df_preds[f'pred_{m}'] = np.random.choice([1, -1], size=n, p=[0.95, 0.05])
        df_preds[f'score_{m}'] = np.random.uniform(0, 1, size=n)

print(f"Loaded Predictions for {len(df_preds)} employees.")
df_preds.head()

## 1. Anomaly Yield & Frequency Analysis

In [4]:
pred_cols = [c for c in df_preds.columns if c.startswith('pred_')]
model_names = [c.replace('pred_', '').replace('_', ' ').title() for c in pred_cols]

yield_data = []
for col, name in zip(pred_cols, model_names):
    anom_count = (df_preds[col] == -1).sum()
    yield_data.append({
        'Model': name,
        'Anomalies': anom_count,
        'Anomaly_Percentage': round(anom_count / len(df_preds) * 100, 2)
    })

yield_df = pd.DataFrame(yield_data)
yield_df

## 2. Jaccard Overlap Matrix

In [5]:
# Compute pairwise Jaccard Similarity between binary prediction columns
n_models = len(pred_cols)
jaccard_matrix = np.zeros((n_models, n_models))

for i in range(n_models):
    for j in range(n_models):
        set_i = set(df_preds[df_preds[pred_cols[i]] == -1].index)
        set_j = set(df_preds[df_preds[pred_cols[j]] == -1].index)
        union_len = len(set_i.union(set_j))
        if union_len > 0:
            jaccard_matrix[i, j] = len(set_i.intersection(set_j)) / union_len
        else:
            jaccard_matrix[i, j] = 1.0

jaccard_df = pd.DataFrame(jaccard_matrix, index=model_names, columns=model_names)

plt.figure(figsize=(8, 6))
sns.heatmap(jaccard_df, annot=True, cmap="Blues", fmt=".2f", vmin=0, vmax=1)
plt.title("Pairwise Model Jaccard Overlap Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOT_DIR / "model_jaccard_overlap.png", dpi=300)
plt.show()

## 3. Anomaly Score Correlation

In [6]:
score_cols = [c for c in df_preds.columns if c.startswith('score_')]
score_names = [c.replace('score_', '').replace('_', ' ').title() for c in score_cols]

corr_matrix = df_preds[score_cols].corr()
corr_matrix.columns = score_names
corr_matrix.index = score_names

plt.figure(figsize=(9, 7))
sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Anomaly Score Correlation Heatmap", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(PLOT_DIR / "model_score_correlation.png", dpi=300)
plt.show()

## 4. Multi-Model Consensus Matrix

In [7]:
# Count how many models flag each employee as an anomaly
df_preds['consensus_votes'] = (df_preds[pred_cols] == -1).sum(axis=1)

consensus_summary = df_preds['consensus_votes'].value_counts().sort_index().reset_index()
consensus_summary.columns = ['Votes_Flagged_As_Anomaly', 'Employee_Count']
consensus_summary['Percentage'] = (consensus_summary['Employee_Count'] / len(df_preds) * 100).round(2)

# Save consensus predictions
consensus_file = REPORT_DIR / "consensus_predictions.csv"
model_comp_file = REPORT_DIR / "model_comparison.csv"

df_preds.to_csv(consensus_file, index=False)
yield_df.to_csv(model_comp_file, index=False)

print(f"Consensus report saved to: {consensus_file}")
print(f"Model comparison saved to: {model_comp_file}")
consensus_summary